## **🔷 `Week - 1` `Day - 1`**

In [1]:
import json, os ,time

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
load_dotenv()

True

In [4]:
client = OpenAI(
    base_url = "https://openrouter.ai/api/v1",
    api_key = os.environ["OPENROUTER_API_KEY"]
)

In [5]:
log_file = "logs/calls.jsonl"

> **USD per 1 million tokens: (input price, output price)**
- Check the real price on the model's OpenRouter page and update this.

In [6]:
PRICES = {
    "openai/gpt-4o-mini" : (.15, .6)
}

MODEL = "openai/gpt-4o-mini"

$$ \LARGE \text {1 - Logging wrapper} $$

In [7]:
def llm(message, model = MODEL, tools = None, max_tokens = 1000):
    """
        Call the model, log model/ tokens / latency / cost to calls.jsonl, return the response.
    """

    kwargs = {'model' : model, 'messages' : message, 'max_tokens' : max_tokens}
    
    if tools:
        kwargs['tools'] = tools

    start = time.perf_counter()
    resp = client.chat.completions.create(**kwargs)
    latency = time.perf_counter() - start

    usage = resp.usage
    price_in, price_out = PRICES.get(model, (0, 0))
    cost = (usage.prompt_tokens * price_in + usage.completion_tokens * price_out) / 1_000_000

    record = {
        'model' : model,
        'input_tokens' : usage.prompt_tokens,
        'output_tokens' : usage.completion_tokens,
        'latency_seconds' : round(latency, 2),
        'cost_usd' : round(cost, 6),
        'finish_reason' : resp.choices[0].finish_reason
    }

    with open(log_file, "a") as f:
        f.write(json.dumps(record) + "\n")

    return resp

$$ \LARGE \text {2 - Tool} $$

- **⚠️ Defined `TOOLS` below is the description the model sees. model does NOT see the Python function.**

In [8]:
def get_weather_data(city : str) -> dict:
    """
        Fake weather data fetcher. model never runs this, our code does.
    """
    return {'city' : city, 'temp_c' : 31, 'condition' : 'sunny'}

TOOLS = [
    {
        'type' : 'function',
        'function' : {
            'name' : 'get_weather_data',
            'description' : 'Get the current weather for a city.',
            'parameters' : {
                'type' : 'object',
                'properties' : {
                    'city' : {'type' : 'string', 'description' : 'City name, e.g. Surat'}
                },
                'required' : ['city']
            }
        }
    }
]

In [9]:
TOOL_FUNCTION = {'get_weather_data': get_weather_data}  # function mapping

$$ \LARGE \text {3 - One full tool-call round trip} $$

In [10]:
def main(user_query):
    messages = [{'role' : 'user', 'content' : user_query}]

    # Call-1 : Model decides whether it needs any tool call or not..?
    r1 = llm(messages, tools = TOOLS)

    print("🔷 RAW RESPONSE 1")
    print(r1.model_dump_json(indent=2))

    msg = r1.choices[0].message

    if not msg.tool_calls:
        print("🔷 Model answered directly, no tool used.")
        print(msg.content)
        return

    # ▣ Put the model's tool request into the history
    messages.append(
        {
            'role' : 'assistant',
            'content' : msg.content,
            'tool_calls' : [
                {
                    'id' : tc.id,
                    'type' : 'function',
                    'function' : {
                        'name' : tc.function.name,
                        'arguments' : tc.function.arguments
                    }
                }
                for tc in msg.tool_calls
            ]
        }
    )

    # ▣ Run each requested tool and send the result back
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)
        print(f"\nModel wants : {tc.function.name}({args})")

        result = TOOL_FUNCTION[tc.function.name](**args)

        messages.append(
            {
                'role' : 'tool',
                'tool_call_id' : tc.id,
                'content' : json.dumps(result)
            }
        )

    # Call-2 : The model reads the tool result and writes the final answer
    r2 = llm(messages, tools=TOOLS)
    print("🔷 FINAL ANSWER")
    print(r2.choices[0].message.content)

In [11]:
main('Current weather condition in loss angeles ?')

🔷 RAW RESPONSE 1
{
  "id": "gen-1792258212-emD2KphIlb8nvNXF15rD",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_WjF3wYXjVhv4m7yszZqlFiLr",
            "function": {
              "arguments": "{\"city\":\"Los Angeles\"}",
              "name": "get_weather_data"
            },
            "type": "function",
            "index": 0
          }
        ],
        "reasoning": null
      },
      "native_finish_reason": "tool_calls"
    }
  ],
  "created": 1792258212,
  "model": "openai/gpt-4o-mini",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": "fp_7b25998d8f",
  "usage": {
    "completion_tokens": 16,
    "prompt_token

In [12]:
main('Which city is the capital of gujarat state ?')

🔷 RAW RESPONSE 1
{
  "id": "gen-1792258213-uIk9WOOBzHKmnTUW4ZbE",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "The capital of Gujarat state is Gandhinagar.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "reasoning": null
      },
      "native_finish_reason": "stop"
    }
  ],
  "created": 1792258213,
  "model": "openai/gpt-4o-mini",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": "fp_7b25998d8f",
  "usage": {
    "completion_tokens": 11,
    "prompt_tokens": 66,
    "total_tokens": 77,
    "completion_tokens_details": {
      "accepted_prediction_tokens": null,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": null,
      "text_tokens": null,
      "image_t

> **How `main()` method works.**


```text

    1. Define memory with first user message
    2. First llm call with user_query and supported tool list
    3. By seeing user_query, model understood that it doesn't have weather data, but there is a method called `get_weather_data` which can help
       it to get weather data, but LLM can't run it, so it response back with request to run the method `get_weather_data` by your side and send result.
    4. We will add this response to LLM's memory with requested method's input structure
    5. we will run this method and store the output again in LLM's memory
    6. Now call LLM again, now since there is output in LLM's memory now LLM will be able to give answer. 

```